# Selección de los 10 niños reservados para la prueba del pipeline
**Settings:** CPU, sin internet. **Inputs:** `ahmed-full-dataset`, `odontify-manifiestos`, salida del notebook `minado1-anotaciones-ahmed`.

**Regla (fijada antes de mirar resultados):**
- Elegibles: `pipeline_pool` (no están en ICDAS ni en segmentación) y con las 5 vistas con retractor.
- Estratos por cajas de caries de Ahmed (D/d) en sus fotos con retractor: **0 cajas → 3 niños; 1–3 → 4; ≥ 4 → 3**.
- Orden aleatorio dentro de cada estrato con semilla fija (2026). Solo se puede vetar por **calidad de imagen**; el veto pasa al siguiente del mismo estrato.

Salidas: `reserva_candidatos.csv` (orden del sorteo) y `seleccion_reserva.html` (fotos con las cajas de Ahmed, veto por calidad y exportación de los 10).

In [ ]:
import os, glob, json, base64, numpy as np, pandas as pd, cv2
IN, W = '/kaggle/input', '/kaggle/working'
SEED, CUPOS = 2026, {'0 cajas': 3, '1-3 cajas': 4, '>=4 cajas': 3}
SUPLENTES = 4
def find(name):
    h = glob.glob(f'{IN}/**/{name}', recursive=True); return h[0] if h else None
IMG = [d for d in glob.glob(f'{IN}/**/Dataset/Images', recursive=True) if os.path.isdir(d)][0]
ch = max((pd.read_csv(p) for p in glob.glob(f'{IN}/**/children.csv', recursive=True)), key=lambda d: d.shape[1])
inv = pd.read_csv(find('ahmed_inventario.csv'))
caj = pd.read_csv(find('ahmed_cajas.csv'))
k2c = {k: r.child_id for r in ch.itertuples() for k in str(r.keys).split(';')}
inv['child_id'] = inv.patient_key.map(k2c)
ret = inv[inv.condition == 'retractors']
vistas = ret.groupby('child_id').view.nunique()
eleg = ch[(ch.pipeline_pool == True) & ch.child_id.map(vistas).eq(5)].copy()
nc = caj[caj.condition == 'retractors'].groupby('child_id').size()
eleg['cajas_ret'] = eleg.child_id.map(nc).fillna(0).astype(int)
eleg['estrato'] = pd.cut(eleg.cajas_ret, [-1, 0, 3, 999], labels=list(CUPOS)).astype(str)
print('Elegibles:', len(eleg)); print(eleg.estrato.value_counts())
rng = np.random.default_rng(SEED); out = []
for e, n in CUPOS.items():
    d = eleg[eleg.estrato == e].sample(frac=1, random_state=SEED).reset_index(drop=True)
    d['orden'] = np.arange(1, len(d) + 1); d['rol'] = np.where(d.orden <= n, 'titular', 'suplente')
    out.append(d.head(n + SUPLENTES))
sel = pd.concat(out, ignore_index=True)
sel.to_csv(f'{W}/reserva_candidatos.csv', index=False)
print(sel[['child_id', 'estrato', 'orden', 'rol', 'cajas_ret', 'keys']].to_string())

In [ ]:
ORDER = ['Frontal', 'Right_Lateral', 'Left_Lateral', 'Maxillary_Occlusal', 'Mandibular']
def thumb(r, boxes, w=360):
    im = cv2.imread(os.path.join(IMG, r.ref_rel))
    if im is None: return ''
    for b in boxes.itertuples():
        cv2.rectangle(im, (int(b.x0), int(b.y0)), (int(b.x1), int(b.y1)), (0, 0, 255), max(2, im.shape[1] // 250))
    s = w / im.shape[1]; im = cv2.resize(im, (w, int(im.shape[0] * s)))
    return base64.b64encode(cv2.imencode('.jpg', im, [cv2.IMWRITE_JPEG_QUALITY, 72])[1]).decode()
cards = []
for r in sel.itertuples():
    fotos = ret[ret.child_id == r.child_id].copy(); fotos['o'] = fotos.view.map(ORDER.index); fotos = fotos.sort_values('o')
    imgs = [dict(view=f.view, b64=thumb(f, caj[caj.ref_rel == f.ref_rel])) for f in fotos.itertuples()]
    cards.append(dict(child=r.child_id, estrato=r.estrato, orden=int(r.orden), rol=r.rol, cajas=int(r.cajas_ret), keys=r.keys, imgs=imgs))
data = json.dumps(dict(cupos=CUPOS, cards=cards))


In [ ]:
HTML = r"""<!doctype html><html lang="es"><head><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1">
<title>Selección reserva pipeline</title>
<style>
:root{--bg:#f7f7f5;--fg:#1f1f1d;--mut:#6b6a66;--card:#fff;--bd:#dddcd6;--acc:#2f6f5e;--bad:#b3402a}
@media (prefers-color-scheme:dark){:root{--bg:#1c1c1a;--fg:#ecebe6;--mut:#a3a29c;--card:#262624;--bd:#3a3936;--acc:#6cc3a6;--bad:#ef8a73}}
body{margin:0;background:var(--bg);color:var(--fg);font:15px/1.5 system-ui,sans-serif}
header{position:sticky;top:0;background:var(--bg);border-bottom:1px solid var(--bd);padding:12px 16px;z-index:2}
h1{font-size:18px;margin:0 0 4px} .mut{color:var(--mut);font-size:13px}
main{padding:16px;max-width:1900px;margin:auto} h2{font-size:16px;margin:24px 0 8px}
.card{background:var(--card);border:1px solid var(--bd);border-radius:10px;padding:10px 12px;margin:10px 0}
.card.vetado{opacity:.45} .card.final{border-color:var(--acc);box-shadow:0 0 0 2px var(--acc) inset}
.top{display:flex;gap:12px;align-items:center;flex-wrap:wrap} .tag{font-size:12px;padding:2px 8px;border-radius:99px;border:1px solid var(--bd)}
.tag.ok{color:var(--acc);border-color:var(--acc)} .imgs{display:flex;gap:6px;overflow-x:auto;margin-top:8px}
.imgs figure{margin:0} .imgs img{height:200px;border-radius:6px;cursor:zoom-in} figcaption{font-size:11px;color:var(--mut)}
button{font:inherit;padding:6px 12px;border-radius:8px;border:1px solid var(--bd);background:var(--card);color:var(--fg);cursor:pointer}
button.primary{background:var(--acc);color:#fff;border-color:var(--acc)} input[type=text]{font:inherit;padding:5px 8px;border-radius:6px;border:1px solid var(--bd);background:var(--card);color:var(--fg);min-width:240px}
#zoom{position:fixed;inset:0;background:rgba(0,0,0,.85);display:none;align-items:center;justify-content:center;z-index:9} #zoom img{max-width:96vw;max-height:94vh}
</style></head><body>
<header><h1>Selección de los 10 niños reservados para la prueba del pipeline</h1>
<div class="mut">Sorteo estratificado (semilla 2026). Solo se veta por <b>calidad de imagen</b>; entra el siguiente suplente del mismo estrato. Cajas rojas = caries anotadas por Ahmed.</div>
<div class="top" style="margin-top:8px"><span id="resumen" class="tag"></span><button class="primary" onclick="exportar()">Exportar los 10 (CSV)</button><button onclick="if(confirm('¿Borrar vetos?')){V={};save();render()}">Reiniciar vetos</button></div></header>
<main id="main"></main><div id="zoom" onclick="this.style.display='none'"><img></div>
<script>
const D = __DATA__; let V = {};
try { V = JSON.parse(localStorage.getItem('reserva_vetos_v1') || '{}') } catch (e) {}
function save(){ try { localStorage.setItem('reserva_vetos_v1', JSON.stringify(V)) } catch (e) {} }
function finales(){ const f = []; for (const [e, n] of Object.entries(D.cupos)) { f.push(...D.cards.filter(c => c.estrato === e && !V[c.child]).sort((a, b) => a.orden - b.orden).slice(0, n)) } return f }
function render(){
  const fin = new Set(finales().map(c => c.child)); const m = document.getElementById('main'); m.innerHTML = '';
  for (const [e, n] of Object.entries(D.cupos)) {
    const h = document.createElement('h2'); h.textContent = `Estrato ${e} — se eligen ${n}`; m.appendChild(h);
    for (const c of D.cards.filter(c => c.estrato === e).sort((a, b) => a.orden - b.orden)) {
      const d = document.createElement('div'); d.className = 'card' + (V[c.child] ? ' vetado' : '') + (fin.has(c.child) ? ' final' : '');
      d.innerHTML = `<div class="top"><b>${c.child}</b><span class="tag">orden ${c.orden}</span><span class="tag">${c.cajas} cajas</span>
        ${fin.has(c.child) ? '<span class="tag ok">ELEGIDO</span>' : ''}<span class="mut">${c.keys}</span>
        <label><input type="checkbox" ${V[c.child] ? 'checked' : ''}> vetar por calidad</label>
        <input type="text" placeholder="motivo (ej. foto borrosa)" value="${(V[c.child] && V[c.child].motivo) || ''}"></div>
        <div class="imgs">${c.imgs.map(i => `<figure><img src="data:image/jpeg;base64,${i.b64}" alt="${i.view}"><figcaption>${i.view}</figcaption></figure>`).join('')}</div>`;
      const cb = d.querySelector('input[type=checkbox]'), tx = d.querySelector('input[type=text]');
      cb.onchange = () => { if (cb.checked) V[c.child] = { motivo: tx.value }; else delete V[c.child]; save(); render() };
      tx.onchange = () => { if (V[c.child]) { V[c.child].motivo = tx.value; save() } };
      d.querySelectorAll('img').forEach(im => im.onclick = () => { const z = document.getElementById('zoom'); z.querySelector('img').src = im.src; z.style.display = 'flex' });
      m.appendChild(d);
    }
  }
  document.getElementById('resumen').textContent = `Elegidos: ${fin.size} / 10 · vetados: ${Object.keys(V).length}`;
}
function exportar(){
  const rows = [['child_id','estrato','orden','cajas_ret','keys','estado','motivo_veto']];
  const fin = new Set(finales().map(c => c.child));
  for (const c of D.cards) rows.push([c.child, c.estrato, c.orden, c.cajas, c.keys, fin.has(c.child) ? 'elegido' : (V[c.child] ? 'vetado' : 'no_usado'), (V[c.child] && V[c.child].motivo) || '']);
  const csv = rows.map(r => r.map(x => `"${String(x).replace(/"/g, '""')}"`).join(',')).join('\n');
  const a = document.createElement('a'); a.href = URL.createObjectURL(new Blob([csv], {type: 'text/csv'})); a.download = 'reserva_pipeline_10.csv'; a.click();
}
render();
</script></body></html>"""
open(f'{W}/seleccion_reserva.html', 'w').write(HTML.replace('__DATA__', data))
print('HTML:', round(os.path.getsize(f'{W}/seleccion_reserva.html') / 1e6, 1), 'MB |', len(cards), 'niños')